# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [29]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [30]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [31]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [32]:
# This function looks rather simpler, because we're taking advantage of the latest Gradio updates

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [33]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [34]:
get_ticket_price("dsfrgrd")

Tool get_ticket_price called for dsfrgrd


'Unknown'

In [35]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [36]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [37]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f" Response from Open AI 1 : {response.choices[0]}")
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Response message from Open AI: {message}")
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        print(f"Final message we are sending to openai : {messages}")
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [38]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    print(f"tool_call : {tool_call}")
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [39]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


 Response from Open AI 1 : Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I'm here to assist you! How can I help you today?", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))
 Response from Open AI 1 : Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FAjG3DcvEy12VarvA9MCVLrK', function=Function(arguments='{"destination_city":"Jaipur"}', name='get_ticket_price'), type='function')]))
Response message from Open AI: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FAjG3DcvEy12VarvA9MCVLrK', function=Function(arguments='{"destination_city":"Jaipur"}', name='get_ticket_price'), type='fu

## ✅ Exercise - To understand How it works:

**A user asks:**
"What would be my insurance premium if I’m 35 years old and male?"

The LLM decides to call the **calculate_insurance_premium** tool.

It sends the request with **age and gender.**

The tool responds with the **premium and message.**

**The LLM replies:**
"Your monthly insurance premium is $250."

```{
  "tool_call": {
    "name": "calculate_insurance_premium",
    "parameters": {
      "age": 35,
      "gender": "male"
    }
  }
}```

In [62]:
calculate_insurance_premium = {
    "name": "calculate_insurance_premium",
    "description": "Calculate the insurance premium based on the customer's age and gender. Call this whenever you need to estimate a customer’s insurance cost, for example when they ask 'How much is my insurance?'",
    "parameters": {
        "type": "object",
        "properties": {
            "age": {
                "type": "integer",
                "description": "The age of the customer in years"
            },
            "gender": {
                "type": "string",
                "description": "The gender of the customer (e.g., 'male', 'female', 'other')"
            }
        },
        "required": ["age", "gender"],
        "additionalProperties": False
    }
}


In [63]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": calculate_insurance_premium}]

In [64]:
def calculate_insurance_premium(age, gender):
    print(f"Tool calculate_insurance_premium called for age: {age}, gender: {gender}")

    base_premium = 100  # base premium in dollars

    # Adjust premium based on age
    if age < 25:
        age_factor = 1.5
    elif age < 40:
        age_factor = 1.2
    elif age < 60:
        age_factor = 1.0
    else:
        age_factor = 1.3

    # Adjust premium based on gender
    if gender.lower() == "male":
        gender_factor = 1.1
    elif gender.lower() == "female":
        gender_factor = 1.0
    else:
        gender_factor = 1.05  # default factor for other/non-specified genders

    premium = base_premium * age_factor * gender_factor
    return f"${premium:.2f}"


In [65]:
system_message = "You are a helpful assistant that calculates insurance premiums based on a customer's age and gender. "
system_message += "Always assume age and gender are enough to provide an accurate premium estimate. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "If you don't know the answer, say so honestly."


In [66]:
def chat_insurance(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f" Response from Open AI 1 : {response.choices[0]}")
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print(f"Response message from Open AI: {message}")
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        print(f"Final message we are sending to openai : {messages}")
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [69]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    print(f"tool_call : {tool_call}")
    arguments = json.loads(tool_call.function.arguments)
    age = arguments.get('age')
    gender = arguments.get('gender')
    premium = calculate_insurance_premium(age, gender)
    response = {
        "role": "tool",
        "content": json.dumps({"age": age, "gender": gender, "premium": premium}),
        "tool_call_id": tool_call.id
    }
    return response, (age, gender)

In [70]:
gr.ChatInterface(fn=chat_insurance, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


 Response from Open AI 1 : Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OluQKL01sKfM10sAQv9zL7wH', function=Function(arguments='{"age":34,"gender":"male"}', name='calculate_insurance_premium'), type='function')]))
Response message from Open AI: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OluQKL01sKfM10sAQv9zL7wH', function=Function(arguments='{"age":34,"gender":"male"}', name='calculate_insurance_premium'), type='function')])
tool_call : ChatCompletionMessageFunctionToolCall(id='call_OluQKL01sKfM10sAQv9zL7wH', function=Function(arguments='{"age":34,"gender":"male"}', name='calculate_insurance_premium'), type='function')
Tool calculate_insurance_premium called for 